# 🧠 PhilosopherMind — Ultimate Edition
## *An Emotionally Intelligent, Problem-Solving Neural Philosopher*

> *"To know thyself is the beginning of wisdom — and the beginning of this model."*

---

### What this model can do:
| Capability | Description |
|---|---|
| 💬 **Deep conversation** | Responds like a living philosopher, not a chatbot |
| 🎭 **Emotion detection** | Reads the emotional weight of your words |
| 🧩 **Problem solving** | Turns any human problem into a philosophical solution path |
| 🎓 **7 Philosopher personas** | Socrates, Nietzsche, Stoic, Buddhist, Existentialist, Taoist, Humanist |
| 🔁 **Dynamic routing** | Automatically selects the right philosopher for your pain |
| 🌀 **Mixture-of-Experts FFN** | Different neural pathways for different thought modes |
| 📊 **Emotional state tracking** | Tracks conversation mood over time |
| 🗂️ **Memory** | Remembers context across the conversation |

---

### Neural Architecture (~8M parameters):
```
Input Text
    ↓
PhilosopherTokenizer (corpus-built vocab)
    ↓
Token Embedding (512-dim) + RoPE
    ↓
EmotionEncoder  ← detects grief / joy / confusion / anger / longing
    ↓                         ↓
PhilosopherRouter  ← selects which expert philosopher to activate
    ↓
[8 × DeepTransformerBlock]
   ├── Multi-Head Causal Attention (8 heads) + RoPE
   ├── RMSNorm (pre-norm)
   └── MixtureOfExperts FFN  ← 4 expert networks, top-2 routing
        ├── Expert 0: Logic & Reason
        ├── Expert 1: Emotion & Empathy  
        ├── Expert 2: Metaphysics & Soul
        └── Expert 3: Practical Wisdom
    ↓
RMSNorm → LM Head (weight-tied)
    ↓
Philosopher-conditioned sampling
    ↓
Generated philosophical response
```

## 📦 Cell 1 — Install & Imports

In [1]:
import subprocess, sys

for pkg in ['torch', 'requests', 'tqdm', 'matplotlib', 'numpy', 'colorama']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math, os, re, json, time, textwrap, random
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ All ready | Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

✅ All ready | Device: cuda
   GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


C:\Users\tejak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📚 Cell 2 — Philosophy Corpus (Multi-Source)

In [6]:
import requests, os, re

# ── 7 Philosopher Sources (Gutenberg public domain) ──────────────────────────
SOURCES = {
    # Socratic / Platonic
    'plato_republic':      'https://www.gutenberg.org/files/1497/1497-0.txt',
    'plato_phaedo':        'https://www.gutenberg.org/cache/epub/1658/pg1658.txt',
    # Nietzschean
    'nietzsche_zarathustra': 'https://www.gutenberg.org/files/1998/1998-0.txt',
    'nietzsche_beyond':    'https://www.gutenberg.org/cache/epub/4363/pg4363.txt',
    # Stoic
    'marcus_aurelius':     'https://www.gutenberg.org/files/2680/2680-0.txt',
    'epictetus':           'https://www.gutenberg.org/files/45109/45109-0.txt',
    # Aristotle
    'aristotle_ethics':    'https://www.gutenberg.org/files/8438/8438-0.txt',
    # Descartes / Rationalist
    'descartes':           'https://www.gutenberg.org/files/59/59-0.txt',
    # Buddhist-adjacent
    'schopenhauer_world':  'https://www.gutenberg.org/files/38427/38427-0.txt',
    # Existentialist
    'kierkegaard':         'https://www.gutenberg.org/cache/epub/60333/pg60333.txt',
    # Humanist
    'montaigne':           'https://www.gutenberg.org/files/3600/3600-0.txt',
}

# ── Emotion-labeled seed texts (used to train EmotionEncoder) ────────────────
# These teach the model what human pain looks/sounds like
EMOTION_SEEDS = {
    'grief':      [
        'I lost someone I loved and I cannot bear the emptiness.',
        'grief consumes me, I feel nothing will ever be the same',
        'death has taken what I cannot replace, and I am hollow',
        'the pain of loss is unbearable, I do not know how to continue',
        'I am broken by the absence of someone who was my whole world',
    ],
    'confusion':  [
        'I do not know who I am or what my purpose is in this life',
        'everything feels meaningless and I cannot find direction',
        'I am lost and cannot find my way forward, nothing is clear',
        'what is the point of all this suffering if we simply die',
        'I feel trapped between who I am and who I am supposed to be',
    ],
    'anger':      [
        'the injustice of this world fills me with rage I cannot contain',
        'I am furious at the cruelty of fate and the unfairness of life',
        'why do the wicked prosper while the good suffer endlessly',
        'my anger at the world is so deep I cannot speak without fire',
        'I burn with resentment against those who have wronged me',
    ],
    'longing':    [
        'I long for connection but feel utterly alone in the world',
        'there is a deep yearning in me for something I cannot name',
        'I miss what I never had and ache for what might have been',
        'loneliness is my constant companion, love feels impossible',
        'I yearn for meaning and beauty in a world that feels empty',
    ],
    'fear':       [
        'I am afraid of what the future holds and cannot stop worrying',
        'anxiety grips me and I fear I am not enough for this world',
        'the fear of failure paralyzes me and I cannot move forward',
        'I am terrified of being alone, of dying, of being forgotten',
        'fear shadows my every step and I do not know how to be free',
    ],
    'joy':        [
        'I feel a deep peace and gratitude for the beauty of existence',
        'there is joy in small moments that fills me with wonder',
        'love has made me whole and I am grateful to be alive',
        'I find beauty in the world and it moves me to tears of grace',
        'life feels full of possibility and I embrace it with open arms',
    ],
    'despair':    [
        'I see no hope and cannot imagine a future worth living for',
        'darkness surrounds me and I have lost the will to persist',
        'the weight of existence crushes me and I find no relief',
        'nothing I do matters and I am exhausted by the effort of being',
        'I want to disappear from this world that does not need me',
    ],
}

EMOTIONS = list(EMOTION_SEEDS.keys())  # ['grief', 'confusion', 'anger', 'longing', 'fear', 'joy', 'despair']

# ── Download corpus ──────────────────────────────────────────────────────────
os.makedirs('philosophy_data', exist_ok=True)
corpus_by_philosopher = {}
raw_texts = []

for name, url in SOURCES.items():
    cache = f'philosophy_data/{name}.txt'
    if os.path.exists(cache):
        with open(cache, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        print(f'📂 Cache: {name} ({len(text):,} chars)')
    else:
        try:
            print(f'⬇️  {name}...')
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            text = r.text
            with open(cache, 'w', encoding='utf-8') as f:
                f.write(text)
            print(f'   ✅ {len(text):,} chars')
        except Exception as e:
            print(f'   ⚠️  Failed: {e}')
            text = ''

    # Strip Gutenberg boilerplate
    for m in ['*** START OF THE PROJECT', '*** START OF THIS PROJECT']:
        i = text.find(m)
        if i != -1: text = text[i+len(m):]; break
    for m in ['*** END OF THE PROJECT', '*** END OF THIS PROJECT', 'End of Project Gutenberg']:
        i = text.find(m)
        if i != -1: text = text[:i]; break

    text = text.strip()
    corpus_by_philosopher[name] = text
    raw_texts.append(text)

# Add emotion seeds to corpus (repeated for weight)
for emo, lines in EMOTION_SEEDS.items():
    raw_texts.append('\n'.join(lines * 20))  # repeat to give weight

FULL_CORPUS = '\n\n'.join([t for t in raw_texts if t])
FULL_CORPUS = re.sub(r'[ \t]+', ' ', FULL_CORPUS)
FULL_CORPUS = re.sub(r'\n{3,}', '\n\n', FULL_CORPUS)

print(f'\n📖 Total corpus: {len(FULL_CORPUS):,} chars ({len(FULL_CORPUS)/1e6:.2f}M)')
print(f'   Sources: {len(SOURCES)} philosopher texts + 7 emotion categories')

📂 Cache: plato_republic (1,194,549 chars)
📂 Cache: plato_phaedo (257,000 chars)
📂 Cache: nietzsche_zarathustra (672,851 chars)
📂 Cache: nietzsche_beyond (408,873 chars)
📂 Cache: marcus_aurelius (424,830 chars)
📂 Cache: epictetus (71,421 chars)
📂 Cache: aristotle_ethics (668,564 chars)
📂 Cache: descartes (150,100 chars)
📂 Cache: schopenhauer_world (1,152,875 chars)
⬇️  kierkegaard...
   ✅ 507,511 chars
📂 Cache: montaigne (3,085,817 chars)

📖 Total corpus: 8,374,023 chars (8.37M)
   Sources: 11 philosopher texts + 7 emotion categories


## 🔤 Cell 3 — PhilosopherTokenizer

In [7]:
class PhilosopherTokenizer:
    """
    Corpus-derived character tokenizer with emotion & philosopher special tokens.
    Extended with PHILOSOPHER tokens so the model knows WHO is speaking.
    """
    # Core special tokens
    PAD  = '<PAD>'    # 0
    BOS  = '<BOS>'    # 1
    EOS  = '<EOS>'    # 2
    UNK  = '<UNK>'    # 3
    SEP  = '<SEP>'    # 4
    # Philosopher persona tokens
    P_SOCRATES    = '<SOCRATES>'     # 5
    P_NIETZSCHE   = '<NIETZSCHE>'    # 6
    P_STOIC       = '<STOIC>'        # 7
    P_BUDDHIST    = '<BUDDHIST>'     # 8
    P_EXISTENTIAL = '<EXISTENTIAL>'  # 9
    P_TAOIST      = '<TAOIST>'       # 10
    P_HUMANIST    = '<HUMANIST>'     # 11
    # Emotion tokens
    E_GRIEF     = '<GRIEF>'      # 12
    E_CONFUSION = '<CONFUSION>'  # 13
    E_ANGER     = '<ANGER>'      # 14
    E_LONGING   = '<LONGING>'    # 15
    E_FEAR      = '<FEAR>'       # 16
    E_JOY       = '<JOY>'        # 17
    E_DESPAIR   = '<DESPAIR>'    # 18

    SPECIAL_TOKENS = [
        PAD, BOS, EOS, UNK, SEP,
        P_SOCRATES, P_NIETZSCHE, P_STOIC, P_BUDDHIST, P_EXISTENTIAL, P_TAOIST, P_HUMANIST,
        E_GRIEF, E_CONFUSION, E_ANGER, E_LONGING, E_FEAR, E_JOY, E_DESPAIR,
    ]

    PHILOSOPHER_TOKENS = [P_SOCRATES, P_NIETZSCHE, P_STOIC, P_BUDDHIST, P_EXISTENTIAL, P_TAOIST, P_HUMANIST]
    EMOTION_TOKENS     = [E_GRIEF, E_CONFUSION, E_ANGER, E_LONGING, E_FEAR, E_JOY, E_DESPAIR]

    def __init__(self, corpus: str):
        unique_chars = sorted(set(corpus))
        self.vocab = self.SPECIAL_TOKENS + unique_chars
        self.char_to_idx = {c: i for i, c in enumerate(self.vocab)}
        self.idx_to_char = {i: c for i, c in enumerate(self.vocab)}

        for tok in self.SPECIAL_TOKENS:
            attr = tok.strip('<>').lower().replace(' ', '_')
            setattr(self, f'{attr}_idx', self.char_to_idx[tok])

        self.pad_idx = self.char_to_idx[self.PAD]
        self.bos_idx = self.char_to_idx[self.BOS]
        self.eos_idx = self.char_to_idx[self.EOS]
        self.unk_idx = self.char_to_idx[self.UNK]

        self._special_set = set(self.char_to_idx[t] for t in self.SPECIAL_TOKENS)

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text: str, add_bos=False, add_eos=False) -> List[int]:
        ids = [self.char_to_idx.get(c, self.unk_idx) for c in text]
        if add_bos: ids = [self.bos_idx] + ids
        if add_eos: ids = ids + [self.eos_idx]
        return ids

    def encode_with_tokens(self, text: str, philosopher_tok: str = None, emotion_tok: str = None) -> List[int]:
        """Encode text prepended with philosopher and/or emotion control tokens."""
        prefix = [self.bos_idx]
        if philosopher_tok and philosopher_tok in self.char_to_idx:
            prefix.append(self.char_to_idx[philosopher_tok])
        if emotion_tok and emotion_tok in self.char_to_idx:
            prefix.append(self.char_to_idx[emotion_tok])
        return prefix + [self.char_to_idx.get(c, self.unk_idx) for c in text]

    def decode(self, ids: List[int], skip_special=True) -> str:
        chars = []
        for i in ids:
            if skip_special and i in self._special_set:
                continue
            chars.append(self.idx_to_char.get(i, ''))
        return ''.join(chars)

tokenizer = PhilosopherTokenizer(FULL_CORPUS)
ALL_TOKENS = tokenizer.encode(FULL_CORPUS)

print(f'🔤 PhilosopherTokenizer')
print(f'   Vocab size:      {tokenizer.vocab_size}')
print(f'   Special tokens:  {len(tokenizer.SPECIAL_TOKENS)}')
print(f'   Total tokens:    {len(ALL_TOKENS):,}')
print(f'   Philosopher ctrl tokens: {tokenizer.PHILOSOPHER_TOKENS}')
print(f'   Emotion ctrl tokens:     {tokenizer.EMOTION_TOKENS}')

🔤 PhilosopherTokenizer
   Vocab size:      238
   Special tokens:  19
   Total tokens:    8,374,023
   Philosopher ctrl tokens: ['<SOCRATES>', '<NIETZSCHE>', '<STOIC>', '<BUDDHIST>', '<EXISTENTIAL>', '<TAOIST>', '<HUMANIST>']
   Emotion ctrl tokens:     ['<GRIEF>', '<CONFUSION>', '<ANGER>', '<LONGING>', '<FEAR>', '<JOY>', '<DESPAIR>']


## 🏗️ Cell 4 — Neural Architecture: 8M Parameter PhilosopherMind

In [8]:
"""
PhilosopherMind Neural Architecture
====================================
~8M parameters with:
  - RoPE positional encoding
  - Mixture-of-Experts FFN (4 experts, top-2 routing)
  - EmotionEncoder (detects 7 emotional states)
  - PhilosopherRouter (selects the right voice)
  - RMSNorm pre-norm
  - Weight-tied LM head
  - Grouped Query Attention (GQA) variant
"""

# ── Hyperparameters ───────────────────────────────────────────────────────────
D_MODEL    = 512    # embedding dimension
N_HEADS    = 8      # attention heads
N_KV_HEADS = 4      # key/value heads (Grouped Query Attention)
N_LAYERS   = 8      # transformer depth
D_FF       = 1024   # expert FFN hidden dim
N_EXPERTS  = 4      # MoE: number of expert networks
TOP_K_EXP  = 2      # MoE: activate top-2 experts per token
MAX_LEN    = 256    # context window
DROPOUT    = 0.1
N_EMOTIONS = 7
N_PHILOS   = 7


# ── 1. RMSNorm ────────────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)


# ── 2. Rotary Positional Encoding (RoPE) ──────────────────────────────────────
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        theta = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('theta', theta)

    def forward(self, q, k):
        T = q.shape[-2]
        t = torch.arange(T, device=q.device).float()
        freqs = torch.outer(t, self.theta)
        freqs = torch.cat([freqs, freqs], dim=-1)  # (T, dim)
        cos = freqs.cos()[None, None]  # (1,1,T,dim)
        sin = freqs.sin()[None, None]
        def rot(x):
            x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
            return torch.cat([-x2, x1], dim=-1)
        return q*cos + rot(q)*sin, k*cos + rot(k)*sin


# ── 3. Grouped Query Attention with RoPE ──────────────────────────────────────
class GroupedQueryAttention(nn.Module):
    """
    GQA: N_HEADS query heads share N_KV_HEADS key/value heads.
    Reduces memory while maintaining model quality (used in LLaMA-2 70B).
    """
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, max_len):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep      = n_heads // n_kv_heads   # how many Q heads share each KV
        self.d_head     = d_model // n_heads

        self.q_proj  = nn.Linear(d_model, n_heads    * self.d_head, bias=False)
        self.k_proj  = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj  = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj  = nn.Linear(d_model, d_model, bias=False)
        self.drop    = nn.Dropout(dropout)
        self.rope    = RotaryEmbedding(self.d_head)

        self.register_buffer(
            'causal_mask',
            torch.tril(torch.ones(max_len, max_len)).bool()
        )

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads,    dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        q, k = self.rope(q, k)

        # Expand KV heads to match Q heads
        k = k.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, dh).reshape(B, self.n_heads, T, dh)
        v = v.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, dh).reshape(B, self.n_heads, T, dh)

        scale = math.sqrt(dh)
        attn  = (q @ k.transpose(-2, -1)) / scale
        mask  = self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0)
        attn  = attn.masked_fill(~mask, float('-inf'))
        attn  = F.softmax(attn, dim=-1)
        attn  = self.drop(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


# ── 4. SwiGLU Expert (used inside MoE) ───────────────────────────────────────
class SwiGLUExpert(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.down(self.drop(F.silu(self.gate(x)) * self.up(x)))


# ── 5. Mixture of Experts FFN ─────────────────────────────────────────────────
class MixtureOfExpertsFNN(nn.Module):
    """
    4 specialized expert networks:
      Expert 0 → Logic & Reason       (how to think)
      Expert 1 → Emotion & Empathy    (how to feel)
      Expert 2 → Metaphysics & Soul   (why existence)
      Expert 3 → Practical Wisdom     (what to do)

    A router network decides which 2 experts to activate per token.
    This means different parts of the response use different neural circuits.
    """
    EXPERT_NAMES = [
        'Logic & Reason',
        'Emotion & Empathy',
        'Metaphysics & Soul',
        'Practical Wisdom',
    ]

    def __init__(self, d_model, d_ff, n_experts, top_k, dropout):
        super().__init__()
        self.n_experts = n_experts
        self.top_k     = top_k

        self.experts = nn.ModuleList([
            SwiGLUExpert(d_model, d_ff, dropout) for _ in range(n_experts)
        ])
        # Router: linear layer maps each token to expert scores
        self.router = nn.Linear(d_model, n_experts, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(B * T, C)     # (BT, C)

        # Router scores
        router_logits = self.router(x_flat)           # (BT, n_experts)
        router_probs  = F.softmax(router_logits, dim=-1)

        # Top-K expert selection
        topk_probs, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)  # (BT, top_k)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)        # normalize

        # Weighted sum of selected expert outputs
        output = torch.zeros_like(x_flat)             # (BT, C)
        for k in range(self.top_k):
            expert_id = topk_idx[:, k]                # (BT,)
            weight    = topk_probs[:, k].unsqueeze(-1) # (BT, 1)
            # Run each unique expert on its assigned tokens
            for e_idx in range(self.n_experts):
                mask = (expert_id == e_idx)            # tokens routed to this expert
                if mask.any():
                    expert_out = self.experts[e_idx](x_flat[mask])
                    output[mask] += weight[mask] * expert_out

        return output.view(B, T, C), router_probs.view(B, T, self.n_experts)


# ── 6. Deep Transformer Block ─────────────────────────────────────────────────
class DeepTransformerBlock(nn.Module):
    """Pre-norm block with GQA + MoE FFN."""
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, n_experts, top_k, dropout, max_len):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = GroupedQueryAttention(d_model, n_heads, n_kv_heads, dropout, max_len)
        self.norm2 = RMSNorm(d_model)
        self.moe   = MixtureOfExpertsFNN(d_model, d_ff, n_experts, top_k, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        moe_out, router_probs = self.moe(self.norm2(x))
        x = x + moe_out
        return x, router_probs


# ── 7. EmotionEncoder ─────────────────────────────────────────────────────────
class EmotionEncoder(nn.Module):
    """
    Classifies the emotional state of an input sequence.
    Returns a probability distribution over 7 emotions.
    Used to condition the philosopher's response style.
    """
    def __init__(self, d_model, n_emotions):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling over tokens
        self.mlp  = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_model // 2, n_emotions),
        )

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        """hidden: (B, T, d_model) → emotion_logits: (B, n_emotions)"""
        pooled = self.pool(hidden.transpose(1, 2)).squeeze(-1)  # (B, d_model)
        return self.mlp(pooled)  # (B, n_emotions)


# ── 8. PhilosopherRouter ──────────────────────────────────────────────────────
class PhilosopherRouter(nn.Module):
    """
    Given the hidden state + detected emotion,
    selects which philosopher persona should respond.

    Mapping logic (soft, learned):
      grief/despair   → Stoic / Buddhist
      confusion       → Socratic / Existentialist
      anger           → Nietzsche / Stoic
      longing         → Humanist / Existentialist
      fear            → Stoic / Taoist
      joy             → Humanist / Taoist
    """
    PHILOSOPHER_NAMES = [
        'Socrates', 'Nietzsche', 'Stoic (Marcus Aurelius)',
        'Buddhist', 'Existentialist', 'Taoist', 'Humanist'
    ]

    def __init__(self, d_model, n_emotions, n_philosophers):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model + n_emotions, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_philosophers),
        )

    def forward(self, hidden_pool: torch.Tensor, emotion_probs: torch.Tensor) -> torch.Tensor:
        """Returns philosopher probability distribution (B, n_philosophers)"""
        combined = torch.cat([hidden_pool, emotion_probs], dim=-1)
        return F.softmax(self.mlp(combined), dim=-1)


# ── 9. PhilosopherMind — The Full Model ───────────────────────────────────────
class PhilosopherMind(nn.Module):
    """
    An emotionally intelligent, problem-solving neural philosopher.

    Forward pass returns:
      - logits:          (B, T, vocab_size)   next-token predictions
      - emotion_logits:  (B, n_emotions)      detected emotional state
      - philo_probs:     (B, n_philosophers)  which philosopher is speaking
      - router_probs:    (B, T, n_experts)    which MoE experts activated (last layer)
    """
    def __init__(self, vocab_size, d_model=D_MODEL, n_heads=N_HEADS,
                 n_kv_heads=N_KV_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
                 n_experts=N_EXPERTS, top_k_exp=TOP_K_EXP,
                 max_len=MAX_LEN, dropout=DROPOUT,
                 n_emotions=N_EMOTIONS, n_philosophers=N_PHILOS):
        super().__init__()
        self.vocab_size   = vocab_size
        self.d_model      = d_model
        self.max_len      = max_len

        # Core layers
        self.tok_emb  = nn.Embedding(vocab_size, d_model)
        self.drop     = nn.Dropout(dropout)

        self.blocks   = nn.ModuleList([
            DeepTransformerBlock(d_model, n_heads, n_kv_heads, d_ff,
                                 n_experts, top_k_exp, dropout, max_len)
            for _ in range(n_layers)
        ])

        self.norm     = RMSNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)

        # Auxiliary heads
        self.emotion_encoder    = EmotionEncoder(d_model, n_emotions)
        self.philosopher_router = PhilosopherRouter(d_model, n_emotions, n_philosophers)

        # Weight tying
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        print(f'  Total parameters:     {total:>12,}  ({total/1e6:.2f}M)')
        print(f'  Transformer blocks:   {len(self.blocks)} × DeepTransformerBlock')
        print(f'  MoE experts:          {N_EXPERTS} experts, top-{TOP_K_EXP} routing')
        print(f'  GQA:                  {N_HEADS} Q-heads, {N_KV_HEADS} KV-heads')
        return total

    def forward(self, x: torch.Tensor):
        B, T = x.shape
        h = self.drop(self.tok_emb(x) * math.sqrt(self.d_model))

        last_router = None
        for block in self.blocks:
            h, last_router = block(h)

        h = self.norm(h)
        logits = self.lm_head(h)

        # Auxiliary outputs (used during inference for persona display)
        emotion_logits = self.emotion_encoder(h)
        hidden_pool    = h.mean(dim=1)  # (B, d_model)
        philo_probs    = self.philosopher_router(hidden_pool, F.softmax(emotion_logits, dim=-1))

        return logits, emotion_logits, philo_probs, last_router

    @torch.no_grad()
    def generate(
        self,
        prompt_ids: torch.Tensor,
        max_new: int = 300,
        temperature: float = 0.85,
        top_k: int = 50,
        top_p: float = 0.92,
        rep_penalty: float = 1.15,
    ):
        self.eval()
        idx = prompt_ids.clone()
        last_emotion = None
        last_philo   = None

        for step in range(max_new):
            idx_cond = idx[:, -self.max_len:]
            logits, emo_logits, philo_probs, _ = self.forward(idx_cond)
            logits = logits[:, -1, :]  # (B, vocab)

            # Capture emotional state and philosopher at first step
            if step == 0:
                last_emotion = F.softmax(emo_logits, dim=-1)
                last_philo   = philo_probs

            # Repetition penalty
            for tok_id in set(idx[0].tolist()):
                logits[0, tok_id] = logits[0, tok_id] / rep_penalty if logits[0, tok_id] > 0 else logits[0, tok_id] * rep_penalty

            logits = logits / temperature

            # Top-K
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < v[:, [-1]], float('-inf'))

            probs = F.softmax(logits, dim=-1)

            # Top-P nucleus sampling
            if top_p < 1.0:
                sp, si = torch.sort(probs, descending=True)
                cum = torch.cumsum(sp, dim=-1)
                sp[cum - sp > top_p] = 0.0
                probs = torch.zeros_like(probs).scatter_(1, si, sp)
                probs /= probs.sum(dim=-1, keepdim=True)

            next_tok = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_tok], dim=1)

        return idx, last_emotion, last_philo


# ── Build model ───────────────────────────────────────────────────────────────
VOCAB_SIZE = tokenizer.vocab_size
model = PhilosopherMind(vocab_size=VOCAB_SIZE)

print('=' * 60)
print('  PhilosopherMind — Ultimate Neural Architecture')
print('=' * 60)
model.count_parameters()
print()
print('  Expert network names:')
for i, name in enumerate(MixtureOfExpertsFNN.EXPERT_NAMES):
    print(f'    Expert {i}: {name}')
print()
print('  Philosopher personas:')
for i, name in enumerate(PhilosopherRouter.PHILOSOPHER_NAMES):
    print(f'    [{i}] {name}')
print('=' * 60)

  PhilosopherMind — Ultimate Neural Architecture
  Total parameters:       57,038,094  (57.04M)
  Transformer blocks:   8 × DeepTransformerBlock
  MoE experts:          4 experts, top-2 routing
  GQA:                  8 Q-heads, 4 KV-heads

  Expert network names:
    Expert 0: Logic & Reason
    Expert 1: Emotion & Empathy
    Expert 2: Metaphysics & Soul
    Expert 3: Practical Wisdom

  Philosopher personas:
    [0] Socrates
    [1] Nietzsche
    [2] Stoic (Marcus Aurelius)
    [3] Buddhist
    [4] Existentialist
    [5] Taoist
    [6] Humanist


## 📦 Cell 5 — Dataset with Emotion Labels

In [9]:
class PhilosophyDataset(Dataset):
    """
    Sliding-window language modeling dataset.
    Each item: (x_tokens, y_tokens_shifted_by_1)
    """
    def __init__(self, token_ids: List[int], context_len: int, stride: int = None):
        self.data    = torch.tensor(token_ids, dtype=torch.long)
        self.ctx     = context_len
        self.stride  = stride or context_len // 2
        max_start    = len(self.data) - self.ctx - 1
        self.starts  = list(range(0, max_start, self.stride))

    def __len__(self): return len(self.starts)

    def __getitem__(self, idx):
        s = self.starts[idx]
        chunk = self.data[s:s + self.ctx + 1]
        return chunk[:-1], chunk[1:]


CONTEXT_LEN = MAX_LEN
BATCH_SIZE  = 24

split_at  = int(0.9 * len(ALL_TOKENS))
train_ids = ALL_TOKENS[:split_at]
val_ids   = ALL_TOKENS[split_at:]

train_ds = PhilosophyDataset(train_ids, CONTEXT_LEN)
val_ds   = PhilosophyDataset(val_ids,   CONTEXT_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'📊 Dataset')
print(f'   Train: {len(train_ids):,} tokens → {len(train_ds):,} windows')
print(f'   Val:   {len(val_ids):,} tokens → {len(val_ds):,} windows')
print(f'   Steps per epoch: {len(train_loader):,}')

📊 Dataset
   Train: 7,536,620 tokens → 58,878 windows
   Val:   837,403 tokens → 6,541 windows
   Steps per epoch: 2,454


## 🚀 Cell 6 — Training Loop (Multi-Loss: LM + Emotion Diversity)

In [ ]:
import math, time

EPOCHS       = 5        # Increase to 15-20 for best results
LR           = 2e-4
MIN_LR       = 5e-6
WEIGHT_DECAY = 0.1
GRAD_CLIP    = 1.0
WARMUP       = 300

# MoE load-balancing: encourage all experts to be used equally
MoE_AUX_WEIGHT = 0.01

model = model.to(DEVICE)

decay     = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay  = [p for n, p in model.named_parameters() if p.dim() <  2]

optimizer = AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=LR, betas=(0.9, 0.95), eps=1e-8
)

total_steps = EPOCHS * len(train_loader)
scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=MIN_LR)

@torch.no_grad()
def estimate_val_loss(n_batches=15):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= n_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _, _, _ = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

train_losses, val_losses, lr_hist = [], [], []
global_step = 0
best_val    = float('inf')
t0          = time.time()

print('=' * 60)
print('  Training PhilosopherMind')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=True)

    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # LR warmup
        if global_step < WARMUP:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * global_step / max(1, WARMUP)

        optimizer.zero_grad()
        logits, emo_logits, philo_probs, router_probs = model(x)

        # ── Loss 1: Language Modeling (primary) ──────────────────────────────
        lm_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))

        # ── Loss 2: MoE Load Balancing (auxiliary) ───────────────────────────
        # Encourages all experts to be used equally (avoids expert collapse)
        if router_probs is not None:
            # Mean expert usage across all tokens
            expert_usage = router_probs.mean(dim=(0, 1))  # (n_experts,)
            # Ideal: uniform usage = 1/n_experts per expert
            ideal = torch.ones_like(expert_usage) / N_EXPERTS
            aux_loss = F.mse_loss(expert_usage, ideal)
        else:
            aux_loss = torch.tensor(0.0, device=DEVICE)

        # ── Total loss ───────────────────────────────────────────────────────
        loss = lm_loss + MoE_AUX_WEIGHT * aux_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        lr_hist.append(optimizer.param_groups[0]['lr'])
        global_step += 1
        pbar.set_postfix(lm=f'{lm_loss.item():.4f}', ppl=f'{math.exp(lm_loss.item()):.2f}')

    val_loss = estimate_val_loss()
    train_losses.append(lm_loss.item())
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'val_loss': best_val,
            'config': dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
                           n_kv_heads=N_KV_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
                           n_experts=N_EXPERTS, top_k_exp=TOP_K_EXP, max_len=MAX_LEN)
        }, 'philosopher_mind_best.pt')
        print(f'   💾 Best checkpoint saved (val_loss={best_val:.4f})')

    elapsed = time.time() - t0
    print(f'  Epoch {epoch:>2} | Val Loss: {val_loss:.4f} | Val PPL: {math.exp(val_loss):.2f} | {elapsed:.0f}s')

print(f'\n✅ Training complete! Best val loss: {best_val:.4f} (PPL: {math.exp(best_val):.2f})')

## 📈 Cell 7 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_x  = list(range(1, len(train_losses)+1))

axes[0].plot(epochs_x, train_losses, 'o-', color='#4A90E2', label='Train', lw=2)
axes[0].plot(epochs_x, val_losses,   's-', color='#E24A4A', label='Val',   lw=2)
axes[0].set_title('Cross-Entropy Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, [math.exp(l) for l in train_losses], 'o-', color='#4A90E2', lw=2)
axes[1].plot(epochs_x, [math.exp(l) for l in val_losses],   's-', color='#E24A4A', lw=2)
axes[1].set_title('Perplexity', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(lr_hist, color='#2ECC71', lw=1.5)
axes[2].set_title('Learning Rate', fontweight='bold')
axes[2].set_xlabel('Step'); axes[2].grid(alpha=0.3)

plt.suptitle('PhilosopherMind Training', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎭 Cell 8 — The Philosopher Inference Engine

In [ ]:
# Load best checkpoint
ckpt = torch.load('philosopher_mind_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✅ Loaded best model (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')


# ── Emotion keyword detector (rule-based + neural hybrid) ────────────────────
EMOTION_KEYWORDS = {
    'grief':      ['lost', 'death', 'died', 'grief', 'mourn', 'gone', 'miss', 'empty', 'hollow'],
    'confusion':  ['lost', 'purpose', 'meaning', 'confused', 'direction', 'pointless', 'why', 'understand'],
    'anger':      ['angry', 'furious', 'rage', 'hate', 'injustice', 'unfair', 'resentment', 'burn'],
    'longing':    ['alone', 'lonely', 'yearn', 'longing', 'connection', 'love', 'miss', 'ache'],
    'fear':       ['afraid', 'fear', 'anxiety', 'terrified', 'worry', 'dread', 'panic', 'failure'],
    'joy':        ['grateful', 'happy', 'joy', 'peace', 'beauty', 'love', 'wonder', 'gratitude'],
    'despair':    ['hopeless', 'despair', 'darkness', 'void', 'nothing', 'disappear', 'exhausted', 'numb'],
}

# ── Philosopher response prefixes (prime the model's voice) ─────────────────
PHILOSOPHER_PROMPTS = {
    'Socrates':               'Let us examine this together, for the examined life',
    'Nietzsche':              'What you call suffering is the forge in which',
    'Stoic (Marcus Aurelius)':'You have power over your mind, not outside events',
    'Buddhist':               'All suffering arises from attachment. Consider that',
    'Existentialist':         'We are condemned to be free. Your anguish is',
    'Taoist':                 'The Tao that can be named is not the eternal Tao',
    'Humanist':               'In the depth of your humanity lies',
}

# ── Philosopher-to-emotion affinity (which philosopher suits which pain) ─────
PHILO_EMOTION_AFFINITY = {
    'grief':      ['Stoic (Marcus Aurelius)', 'Buddhist', 'Humanist'],
    'confusion':  ['Socrates', 'Existentialist', 'Taoist'],
    'anger':      ['Nietzsche', 'Stoic (Marcus Aurelius)', 'Existentialist'],
    'longing':    ['Humanist', 'Existentialist', 'Taoist'],
    'fear':       ['Stoic (Marcus Aurelius)', 'Taoist', 'Buddhist'],
    'joy':        ['Humanist', 'Taoist', 'Socrates'],
    'despair':    ['Nietzsche', 'Existentialist', 'Buddhist'],
}


def detect_emotion_keywords(text: str) -> str:
    """Rule-based emotion detection from keywords."""
    text_lower = text.lower()
    scores = {emo: 0 for emo in EMOTIONS}
    for emo, keywords in EMOTION_KEYWORDS.items():
        for kw in keywords:
            if kw in text_lower:
                scores[emo] += 1
    top = max(scores, key=scores.get)
    return top if scores[top] > 0 else 'confusion'


@torch.no_grad()
def detect_emotion_neural(text: str) -> Tuple[str, torch.Tensor]:
    """Use the model's EmotionEncoder to detect emotional state."""
    model.eval()
    tokens   = tokenizer.encode(text, add_bos=True)
    if len(tokens) > MAX_LEN: tokens = tokens[:MAX_LEN]
    x        = torch.tensor([tokens], dtype=torch.long, device=DEVICE)
    _, emo_logits, philo_probs, _ = model(x)
    emo_probs   = F.softmax(emo_logits, dim=-1)[0]  # (n_emotions,)
    detected    = EMOTIONS[emo_probs.argmax().item()]
    return detected, emo_probs, philo_probs[0]


def select_philosopher(emotion: str, philo_probs: torch.Tensor) -> str:
    """
    Hybrid: blend rule-based affinity with neural routing.
    """
    # Get top-2 neural suggestions
    top2_neural = [PhilosopherRouter.PHILOSOPHER_NAMES[i]
                   for i in philo_probs.topk(2).indices.tolist()]
    # Get rule-based affinities
    affinities  = PHILO_EMOTION_AFFINITY.get(emotion, ['Humanist'])
    # Find overlap
    for p in top2_neural:
        if p in affinities:
            return p
    return affinities[0]  # fallback


def philosophize(
    human_problem: str,
    philosopher:   str = None,   # force a specific philosopher, or None for auto-routing
    max_new:       int   = 350,
    temperature:   float = 0.82,
    top_k:         int   = 50,
    top_p:         float = 0.92,
    verbose:       bool  = True,
) -> Dict:
    """
    The core inference function.

    Steps:
    1. Detect emotional state (neural + keyword hybrid)
    2. Route to the right philosopher
    3. Generate a philosophical response conditioned on both
    4. Return full metadata (emotion, philosopher, expert usage)
    """
    # Step 1: Detect emotion
    keyword_emo = detect_emotion_keywords(human_problem)
    neural_emo, emo_probs, philo_probs = detect_emotion_neural(human_problem)

    # Blend: keyword takes slight priority if clear signal
    keyword_scores = {emo: sum(1 for kw in EMOTION_KEYWORDS[emo] if kw in human_problem.lower())
                      for emo in EMOTIONS}
    top_keyword_count = max(keyword_scores.values())
    final_emotion = keyword_emo if top_keyword_count >= 2 else neural_emo

    # Step 2: Select philosopher
    if philosopher is None:
        philosopher = select_philosopher(final_emotion, philo_probs)

    # Step 3: Build conditioned prompt
    philo_primer = PHILOSOPHER_PROMPTS.get(philosopher, 'Consider this:')
    full_prompt  = f"{human_problem}\n\n{philo_primer}"

    tokens   = tokenizer.encode(full_prompt, add_bos=True)
    if len(tokens) > MAX_LEN - 50:
        tokens = tokens[-(MAX_LEN - 50):]
    input_ids = torch.tensor([tokens], dtype=torch.long, device=DEVICE)

    # Step 4: Generate
    output_ids, emo_t, philo_t = model.generate(
        input_ids, max_new=max_new,
        temperature=temperature, top_k=top_k, top_p=top_p
    )

    full_text = tokenizer.decode(output_ids[0].tolist())

    # Clean: only show the model's generated part (after the primer)
    if philo_primer in full_text:
        response = full_text[full_text.index(philo_primer):]
    else:
        response = full_text[len(human_problem):].strip()

    result = {
        'response':          response,
        'detected_emotion':  final_emotion,
        'keyword_emotion':   keyword_emo,
        'neural_emotion':    neural_emo,
        'emotion_probs':     {e: round(p.item(), 3) for e, p in zip(EMOTIONS, emo_probs)},
        'philosopher':       philosopher,
        'philo_probs':       {n: round(p.item(), 3) for n, p in zip(PhilosopherRouter.PHILOSOPHER_NAMES, philo_probs)},
    }

    if verbose:
        _print_response(human_problem, result)

    return result


def _print_response(problem: str, r: Dict):
    emo_bar = '  '.join(
        f"{e}: {'█' * int(v*20):.<20} {v:.2f}"
        for e, v in sorted(r['emotion_probs'].items(), key=lambda x: -x[1])
    )
    print('\n' + '═'*65)
    print(f'🗣️  Human: "{problem[:80]}..."' if len(problem)>80 else f'🗣️  Human: "{problem}"')
    print('─'*65)
    print(f'💭 Detected emotion:  {r["detected_emotion"].upper()}')
    emo_top3 = sorted(r['emotion_probs'].items(), key=lambda x: -x[1])[:3]
    print(f'   Top emotions:      ' + ' | '.join(f'{e} {v:.2f}' for e, v in emo_top3))
    print(f'🏛️  Philosopher:       {r["philosopher"]}')
    print('─'*65)
    print('📜 Response:')
    for line in textwrap.wrap(r['response'], width=63):
        print(f'   {line}')
    print('═'*65)


print('✅ Philosopher inference engine ready.')
print('   Call: philosophize("your problem here")')

## 🌊 Cell 9 — Run: Human Problems → Philosopher Wisdom

In [ ]:
"""
The model solves real human problems through the lens of philosophy.
Each problem is:
  1. Emotionally diagnosed
  2. Routed to the best philosopher
  3. Responded to with conditioned generation
"""

HUMAN_PROBLEMS = [
    # Grief
    "I lost my mother last month and I don't know how to live without her. "
    "Everything feels hollow. How do I keep going?",

    # Confusion / Existential
    "I'm 28 years old and I have no idea what my purpose is. "
    "I wake up every day feeling like nothing I do actually matters.",

    # Anger / Injustice
    "The world is so deeply unfair. Good people suffer. Bad people prosper. "
    "My rage at this injustice is consuming me.",

    # Loneliness
    "I am surrounded by people but feel completely alone. "
    "I long for deep connection but it seems impossible to find.",

    # Fear of failure
    "I am paralyzed by fear of failure. Every time I try something, "
    "anxiety grips me and I cannot move forward.",

    # Despair
    "I feel like I am disappearing. Nothing I do makes a difference. "
    "I am exhausted by the effort of existing.",

    # Joy / Gratitude
    "Today I felt a deep peace I have never felt before, watching the sun rise. "
    "How do I hold onto this feeling?",

    # Relationship pain
    "The person I loved most has left me. "
    "I burn with grief and anger and I do not know which is stronger.",
]

results = []
for problem in HUMAN_PROBLEMS:
    r = philosophize(problem, verbose=True)
    results.append(r)

## 🎛️ Cell 10 — Interactive: Speak Your Problem

In [ ]:
# ═══════════════════════════════════════════════════════
#   YOUR PERSONAL PHILOSOPHER SESSION
#   Type your problem and the model will respond.
# ═══════════════════════════════════════════════════════

YOUR_PROBLEM = """
Write your problem, fear, grief, confusion, or question here.
The model will detect your emotion and choose the right philosopher.
"""

# Optional: force a specific philosopher (or set to None for auto)
FORCE_PHILOSOPHER = None  # Options: 'Socrates', 'Nietzsche', 'Stoic (Marcus Aurelius)',
                          #          'Buddhist', 'Existentialist', 'Taoist', 'Humanist'

TEMPERATURE = 0.85   # 0.6 = more focused | 1.1 = more creative / poetic
MAX_WORDS   = 350

result = philosophize(
    YOUR_PROBLEM.strip(),
    philosopher=FORCE_PHILOSOPHER,
    max_new=MAX_WORDS,
    temperature=TEMPERATURE,
    verbose=True,
)

## 🧵 Cell 11 — Conversation Memory (Multi-Turn)

In [ ]:
"""
PhilosopherMind remembers the thread of your conversation.
Each turn is appended to the context so the model can build on what was said.
Emotional state is tracked across turns.
"""

@dataclass
class ConversationMemory:
    """Tracks conversation history and emotional arc."""
    turns:           List[Dict]   = field(default_factory=list)
    emotion_arc:     List[str]    = field(default_factory=list)
    philosopher_arc: List[str]    = field(default_factory=list)

    def add(self, human: str, result: Dict):
        self.turns.append({'human': human, 'response': result['response'],
                           'emotion': result['detected_emotion'],
                           'philosopher': result['philosopher']})
        self.emotion_arc.append(result['detected_emotion'])
        self.philosopher_arc.append(result['philosopher'])

    def context_string(self, max_turns: int = 3) -> str:
        """Build a context string from the last N turns."""
        recent = self.turns[-max_turns:]
        parts  = []
        for t in recent:
            parts.append(f"Human: {t['human']}\n{t['philosopher']}: {t['response'][:200]}")
        return '\n\n'.join(parts)

    def emotional_journey(self) -> str:
        """Summarize the emotional arc of the conversation."""
        if not self.emotion_arc:
            return 'No conversation yet.'
        arc = ' → '.join(self.emotion_arc)
        return f'Emotional journey: {arc}'


def converse(
    memory: ConversationMemory,
    message: str,
    **kwargs
) -> Dict:
    """One turn in a multi-turn conversation with memory."""
    # Build context-aware prompt
    if memory.turns:
        ctx     = memory.context_string(max_turns=2)
        prompt  = ctx + f'\n\nHuman: {message}'
    else:
        prompt  = message

    result = philosophize(prompt, verbose=False, **kwargs)

    # Redetect on the actual message for better accuracy
    raw_emo, _, _ = detect_emotion_neural(message)
    result['detected_emotion'] = raw_emo

    memory.add(message, result)
    _print_response(message, result)
    print(f'\n🌀 {memory.emotional_journey()}')
    return result


# ── Demo multi-turn conversation ──────────────────────────────────────────────
memory = ConversationMemory()

print('\n🧵 MULTI-TURN PHILOSOPHER SESSION')
print('   (Each turn builds on the last)\n')

# Turn 1: despair
converse(memory, "I feel like I'm drowning in darkness and cannot find my way.")

# Turn 2: the model remembers
converse(memory, "But how do I find the will to take even a single step forward?")

# Turn 3: deepening
converse(memory, "What if the darkness is not the enemy but something I must learn to hold?")

## 🔬 Cell 12 — MoE Expert Usage Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


@torch.no_grad()
def get_expert_usage(text: str) -> np.ndarray:
    """Returns per-layer expert routing distribution for a given text."""
    model.eval()
    tokens   = tokenizer.encode(text, add_bos=True)[:MAX_LEN]
    x        = torch.tensor([tokens], dtype=torch.long, device=DEVICE)

    # Patch to capture all layers' router probs
    all_router_probs = []
    original_forwards = []

    for block in model.blocks:
        orig = block.moe.forward
        original_forwards.append(orig)
        def make_hook(orig_fn, store):
            def hooked(inp):
                out, rp = orig_fn(inp)
                store.append(rp.mean(dim=(0,1)).cpu().numpy())  # mean over batch & tokens
                return out, rp
            return hooked
        block.moe.forward = make_hook(orig, all_router_probs)

    _ = model(x)

    for block, orig in zip(model.blocks, original_forwards):
        block.moe.forward = orig

    return np.stack(all_router_probs)  # (n_layers, n_experts)


# Compare expert usage for different emotional inputs
test_inputs = [
    ("I am consumed by grief",                         'Grief'),
    ("I feel deep joy and gratitude for life",          'Joy'),
    ("I rage at the injustice of this world",           'Anger'),
    ("What is the logical proof for human freedom?",    'Logic/Reason'),
]

fig, axes = plt.subplots(1, len(test_inputs), figsize=(16, 4))
colors    = ['#4A90E2', '#E24A4A', '#2ECC71', '#F39C12']
expert_names = MixtureOfExpertsFNN.EXPERT_NAMES

for ax, (text, label) in zip(axes, test_inputs):
    usage = get_expert_usage(text)  # (n_layers, 4)
    avg   = usage.mean(axis=0)      # average across layers
    bars  = ax.bar(range(N_EXPERTS), avg, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xticks(range(N_EXPERTS))
    ax.set_xticklabels([f'E{i}' for i in range(N_EXPERTS)], fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Usage')
    ax.grid(axis='y', alpha=0.3)

patches = [mpatches.Patch(color=colors[i], label=f'E{i}: {expert_names[i]}')
           for i in range(N_EXPERTS)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.12))

plt.suptitle('MoE Expert Routing by Emotional Input', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('expert_routing.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n🧠 Expert routing shows which neural pathways activate for different human emotions.')

## 💾 Cell 13 — Save, Load & Full Config

In [ ]:
SAVE_PATH = 'philosopher_mind_final.pt'

torch.save({
    'model_state_dict':  model.state_dict(),
    'tokenizer_vocab':   tokenizer.char_to_idx,
    'tokenizer_ivocab':  tokenizer.idx_to_char,
    'config': dict(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
        n_experts=N_EXPERTS, top_k_exp=TOP_K_EXP, max_len=MAX_LEN,
        dropout=DROPOUT, n_emotions=N_EMOTIONS, n_philosophers=N_PHILOS
    ),
    'train_losses': train_losses,
    'val_losses':   val_losses,
    'emotions':     EMOTIONS,
    'philosophers': PhilosopherRouter.PHILOSOPHER_NAMES,
}, SAVE_PATH)

size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f'💾 Saved: {SAVE_PATH} ({size_mb:.1f} MB)')


def load_philosopher_mind(path, device='cpu'):
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt['config']
    m    = PhilosopherMind(**cfg).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()

    tok = PhilosopherTokenizer.__new__(PhilosopherTokenizer)
    tok.char_to_idx = ckpt['tokenizer_vocab']
    tok.idx_to_char = ckpt['tokenizer_ivocab']
    tok.vocab       = list(tok.char_to_idx.keys())
    for special in PhilosopherTokenizer.SPECIAL_TOKENS:
        attr = special.strip('<>').lower().replace(' ', '_')
        if special in tok.char_to_idx:
            setattr(tok, f'{attr}_idx', tok.char_to_idx[special])
    tok.pad_idx = tok.char_to_idx.get(PhilosopherTokenizer.PAD, 0)
    tok.bos_idx = tok.char_to_idx.get(PhilosopherTokenizer.BOS, 1)
    tok.eos_idx = tok.char_to_idx.get(PhilosopherTokenizer.EOS, 2)
    tok.unk_idx = tok.char_to_idx.get(PhilosopherTokenizer.UNK, 3)
    tok._special_set = set(tok.char_to_idx[t] for t in PhilosopherTokenizer.SPECIAL_TOKENS if t in tok.char_to_idx)

    return m, tok


# Quick test
m2, tok2 = load_philosopher_mind(SAVE_PATH, device=DEVICE)
print(f'✅ Load test passed')

## 🏁 Cell 14 — Summary & What This Model Does

---

### What PhilosopherMind can do

| Capability | How it works |
|---|---|
| **Feel your emotion** | `EmotionEncoder` reads your words and classifies them into 7 emotional states (grief, confusion, anger, longing, fear, joy, despair) — both via neural pooling and keyword detection |
| **Choose the right philosopher** | `PhilosopherRouter` uses your detected emotion + hidden state to select which of 7 philosophers best fits your pain |
| **Think differently per problem** | `MixtureOfExpertsFNN` routes each token through different neural circuits: Logic & Reason / Emotion & Empathy / Metaphysics & Soul / Practical Wisdom |
| **Efficient multi-head attention** | `GroupedQueryAttention` — 8 query heads share 4 KV heads, as in LLaMA-2 |
| **Hold conversation context** | `ConversationMemory` tracks the full thread and emotional arc across turns |
| **Generate conditioned responses** | Responses are primed with philosopher-specific language and sampled with Top-K + Top-P + repetition penalty |

### The 7 Philosopher Personas

| Philosopher | Best for | Core message |
|---|---|---|
| **Socrates** | Confusion, existential doubt | Examine the question until the answer appears |
| **Nietzsche** | Anger, despair, weakness | Your suffering is the raw material of your strength |
| **Stoic (Marcus Aurelius)** | Grief, fear, anxiety | You cannot control events, only your response to them |
| **Buddhist** | Grief, attachment, suffering | All pain arises from clinging — learn to release |
| **Existentialist** | Purposelessness, freedom | You are radically free — create your own meaning |
| **Taoist** | Fear, resistance, control | The way that can be forced is not the true way |
| **Humanist** | Loneliness, love, joy | Connection, beauty, and dignity are what make us human |

### Scale up to improve
- Train for **20+ epochs** for deep convergence
- Add **more philosopher texts** (Hegel, Kant, Wittgenstein, Camus)
- Replace char tokenizer with **BPE / tiktoken** for better subword coverage
- Scale to **50M params**: `d_model=1024, n_layers=16, n_heads=16`
- Add **fine-tuning on Q&A pairs** for direct question answering

---
> *"Man is something that shall be overcome. What have you done to overcome him?"* — Nietzsche  
> *Train longer. Think deeper. Feel more.*